In [1]:
!pip install -q accelerate bitsandbytes transformers qwen-vl-utils pillow torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 56.6 MB/s eta 0:00:00


In [2]:
import os
import json
import time
import random
import torch
from typing import List, Dict, Any, Optional, Tuple
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

class WOAMRawReader:
    def __init__(self, file_path: str):
        self.file_path = file_path
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Không tìm thấy file WOAM: {file_path}")

    def load_records(self, category: Optional[str] = None, split: Optional[str] = None, limit: Optional[int] = None) -> List[Dict[str, Any]]:
        records = []
        with open(self.file_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                item = json.loads(line)

                if category and item.get("pt", "").upper() != category.upper():
                    continue

                is_dev, is_test = item.get("is_dev", False), item.get("is_test", False)
                if split == "dev" and not is_dev: continue
                elif split == "test" and not is_test: continue
                elif split == "train" and (is_dev or is_test): continue

                title = item.get("title", "").strip()
                bullet_points = item.get("bullet_point", [])
                bullet_text = " ".join([str(bp).strip() for bp in bullet_points if bp]) if isinstance(bullet_points, list) else str(bullet_points).strip()

                records.append({
                    "asin": item.get("asin", ""),
                    "category": item.get("pt", ""),
                    "title": title,
                    "bullet_point": bullet_text,
                    "raw": item
                })
                if limit and len(records) >= limit:
                    break
        return records

class MSITDataPreparator:
    def __init__(self, records: List[Dict], image_dir: str):
        self.records = records
        self.image_dir = image_dir

    def _filter_noise(self) -> List[Dict]:
        valid_records = []
        for r in self.records:
            img_path = os.path.join(self.image_dir, f"{r['asin']}.jpg")
            if not os.path.exists(img_path):
                continue
            
            text_length = len(r['title'].split()) + len(r['bullet_point'].split())
            if text_length < 10:
                continue
                
            r['image_path'] = img_path
            valid_records.append(r)
        return valid_records

    def prepare_splits(self) -> Tuple[List[Dict], List[Dict], List[Dict]]:
        cleaned_records = self._filter_noise()
        
        cat_dict = {}
        for r in cleaned_records:
            cat = r['category']
            if cat not in cat_dict:
                cat_dict[cat] = []
            cat_dict[cat].append(r)

        for cat in cat_dict:
            random.shuffle(cat_dict[cat])

        balanced_pool = []
        while cat_dict:
            for cat in list(cat_dict.keys()):
                if cat_dict[cat]:
                    balanced_pool.append(cat_dict[cat].pop(0))
                else:
                    del cat_dict[cat]

        test_pool = balanced_pool[:1000]
        agtd_pool = balanced_pool[1000:2000]
        cttd_pool = balanced_pool[2000:2300]

        all_categories = list(set(r['category'] for r in cleaned_records))
        for i, item in enumerate(cttd_pool):
            if i < len(cttd_pool) // 2:
                item['cttd_label'] = "yes"
                item['attribute_to_judge'] = "target_attribute_placeholder" # Bạn cần thay thế bằng logic bốc thuộc tính thật
            else:
                item['cttd_label'] = "no"
                other_cats = [c for c in all_categories if c != item['category']]
                random_fake_cat = random.choice(other_cats) if other_cats else "Other"
                item['attribute_to_judge'] = f"Fake attribute from {random_fake_cat}"

        return test_pool, agtd_pool, cttd_pool

In [3]:
class QwenLocalDataGenerator:
    def __init__(self, model_id: str = "Qwen/Qwen2.5-VL-7B-Instruct"):
        print("Đang tải mô hình Qwen2.5-VL-7B-Instruct (4-bit)... Vui lòng đợi.")
        
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16
        )
        
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            quantization_config=quantization_config,
            device_map="auto"
        )
        self.processor = AutoProcessor.from_pretrained(model_id)
        print("Tải mô hình hoàn tất! Sẵn sàng sinh dữ liệu.")

    def _clean_json_output(self, text: str) -> str:
        text = text.strip()
        if text.startswith("```json"): text = text[7:]
        elif text.startswith("```"): text = text[3:]
        if text.endswith("```"): text = text[:-3]
        return text.strip()

    def _generate(self, prompt: str, image_path: Optional[str] = None, is_json: bool = False) -> str:
        content = []
        if image_path and os.path.exists(image_path):
            content.append({"type": "image", "image": image_path})
            
        content.append({"type": "text", "text": prompt})
        messages = [{"role": "user", "content": content}]

        text_prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = self.processor(
            text=[text_prompt],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to("cuda")

        with torch.no_grad():
            generated_ids = self.model.generate(
                **inputs, 
                max_new_tokens=512, 
                temperature=0.2
            )
            
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = self.processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        if is_json:
            return self._clean_json_output(output_text)
        return output_text.strip()

    def generate_agtd_sample(self, title: str, bullet_point: str, image_path: str) -> Dict[str, Any]:
        text_prompt = (
            "You are an expert in e-commerce. Extract product attributes and their values from the following text.\n"
            f"Title: {title}\nBullet points: {bullet_point}\n\n"
            "Output strictly in JSON format where keys are attribute names and values are attribute values. "
            "Ensure all JSON keys are in lowercase snake_case format (e.g., compatible_phone_model)."
        )
        text_attributes = self._generate(prompt=text_prompt, is_json=True)
        
        image_attributes = "{}"
        if os.path.exists(image_path):
            img_prompt = (
                "Extract visible product attributes (e.g., color, shape, packaging type) directly from this image.\n"
                "Output strictly in JSON format where keys are attribute names and values are attribute values. "
                "Ensure all JSON keys are in lowercase snake_case format."
            )
            image_attributes = self._generate(prompt=img_prompt, image_path=image_path, is_json=True)

        merge_prompt = (
            "Merge the following two JSON objects containing product attributes. Remove duplicates and keep the most descriptive values.\n"
            f"Text attributes: {text_attributes}\nImage attributes: {image_attributes}\n\n"
            "Output the merged result strictly in JSON format. Do not include any other text."
        )
        merged_attributes = self._generate(prompt=merge_prompt, is_json=True)
        
        try: parsed_attributes = json.loads(merged_attributes)
        except: parsed_attributes = {}

        return {"title": title, "bullet_point": bullet_point, "image_path": image_path, "attributes": parsed_attributes}

    def generate_cttd_sample(self, title: str, bullet_point: str, image_path: str, target_attribute: str) -> Dict[str, Any]:
        prompt = (
            f"Evaluate if the product has the attribute: '{target_attribute}'.\n"
            f"Title: {title}\nBullet points: {bullet_point}\n\n"
            "Provide your reasoning strictly in the following 5 steps:\n"
            "Step 1: Determine the type of product based on pictures and text information.\n"
            "Step 2: Analyze the meaning of the attribute and judge whether it matches the product type based on common sense.\n"
            "Step 3: Check if the product exhibits this attribute from the image.\n"
            "Step 4: Check if the product exhibits this attribute from the text description and verify exact value if present.\n"
            "Step 5: Summarize the conclusions and draw the final decision (yes or no).\n"
        )
        
        cot_response = self._generate(prompt=prompt, image_path=image_path if os.path.exists(image_path) else None)
        
        decision = "no"
        if cot_response:
            decision = "yes" if "yes" in cot_response.lower().split("\n")[-1] else "no"

        return {"title": title, "bullet_point": bullet_point, "image_path": image_path, "target_attribute": target_attribute, "cot_response": cot_response, "decision": decision}

In [4]:
def save_test_set(test_pool: list, output_file: str):
    with open(output_file, "w", encoding="utf-8") as f:
        for item in test_pool:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"Đã lưu {len(test_pool)} mẫu Test Set.")

def generate_and_save_sft_labels(pool: list, output_file: str, mode: str, generator: QwenLocalDataGenerator):
    processed_asins = set()
    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    processed_asins.add(json.loads(line).get("asin", ""))
    
    print(f"\nBắt đầu sinh nhãn {mode.upper()}... (Đã có sẵn {len(processed_asins)}/{len(pool)} mẫu)")

    with open(output_file, "a", encoding="utf-8") as f:
        for idx, item in enumerate(pool):
            asin = item.get("asin", "")
            if asin in processed_asins:
                continue
            
            title = item.get("title", "")
            bp = item.get("bullet_point", "")
            img_path = item.get("image_path", "")

            print(f"[{idx+1}/{len(pool)}] Suy luận cục bộ ASIN: {asin}")
            
            try:
                if mode == "agtd":
                    api_result = generator.generate_agtd_sample(title, bp, img_path)
                    item["attributes"] = api_result.get("attributes", {})
                
                elif mode == "cttd":
                    attr = item.get("attribute_to_judge", "")
                    api_result = generator.generate_cttd_sample(title, bp, img_path, attr)
                    item["cot_response"] = api_result.get("cot_response", "")
                    item["decision"] = api_result.get("decision", item.get("cttd_label"))

                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                # Xóa cache GPU định kỳ để tránh tràn VRAM nếu chạy thời gian dài
                if idx % 10 == 0:
                    torch.cuda.empty_cache()
                
            except Exception as e:
                print(f"[Lỗi] Bỏ qua ASIN {asin}: {e}")

# ================= ĐƯỜNG DẪN MÔI TRƯỜNG KAGGLE =================

# TODO: Thay đổi 'YOUR_DATASET_NAME' thành tên dataset bạn upload lên Kaggle
BASE_INPUT_DIR = "/kaggle/input/datasets/kunzero106/msit-open-world-dataset"
BASE_CHECKPOINT_DIR = "/kaggle/input/datasets/kunzero106/msit-checkpoint-files"
BASE_WORKING_DIR = "/kaggle/working/"

raw_file = os.path.join(BASE_INPUT_DIR, "datasets", "woam", "corpus_with_seeds.jsonl")
image_dir = os.path.join(BASE_INPUT_DIR, "datasets", "images", "woam")

# Các file đầu ra lưu vào /kaggle/working để không bị mất khi session kết thúc
checkpoint_file = os.path.join(BASE_CHECKPOINT_DIR, "splits_checkpoint.json")
test_set_file = os.path.join(BASE_WORKING_DIR, "test_set.jsonl")
agtd_out_file = os.path.join(BASE_WORKING_DIR, "agtd_train.jsonl")
cttd_out_file = os.path.join(BASE_WORKING_DIR, "cttd_train.jsonl")

if __name__ == "__main__":
    if os.path.exists(checkpoint_file):
        print("Đang tải lại dữ liệu từ checkpoint...")
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            splits = json.load(f)
            test_pool, agtd_pool, cttd_pool = splits.get("test", []), splits.get("agtd", []), splits.get("cttd", [])
    else:
        print("Tạo checkpoint và chia tập dữ liệu mới...")
        reader = WOAMRawReader(raw_file)
        raw_records = reader.load_records()
        
        preparator = MSITDataPreparator(raw_records, image_dir)
        test_pool, agtd_pool, cttd_pool = preparator.prepare_splits()
        
        with open(checkpoint_file, "w", encoding="utf-8") as f:
            json.dump({"test": test_pool, "agtd": agtd_pool, "cttd": cttd_pool}, f, ensure_ascii=False, indent=2)
        
        save_test_set(test_pool, test_set_file)
    
    generator = QwenLocalDataGenerator()

    #generate_and_save_sft_labels(agtd_pool, agtd_out_file, mode="agtd", generator=generator)
    generate_and_save_sft_labels(cttd_pool, cttd_out_file, mode="cttd", generator=generator)
    
    print("\nHuấn luyện dữ liệu Local GPU hoàn tất. Hãy tải các file jsonl từ mục Output về máy.")

Đang tải lại dữ liệu từ checkpoint...


Đang tải mô hình Qwen2.5-VL-7B-Instruct (4-bit)... Vui lòng đợi.


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

... (7 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

Tải mô hình hoàn tất! Sẵn sàng sinh dữ liệu.

Bắt đầu sinh nhãn CTTD... (Đã có sẵn 0/300 mẫu)
[1/300] Suy luận cục bộ ASIN: B08L7126H8
[2/300] Suy luận cục bộ ASIN: B07XXBCP8V


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[3/300] Suy luận cục bộ ASIN: B09WJLJQ29
[4/300] Suy luận cục bộ ASIN: B09YXWZK1R
[5/300] Suy luận cục bộ ASIN: B001P3XE6S
[6/300] Suy luận cục bộ ASIN: B08KSXTCTS
[7/300] Suy luận cục bộ ASIN: B076F7DT6S
[8/300] Suy luận cục bộ ASIN: B01LX99851
[9/300] Suy luận cục bộ ASIN: B00B8YT5EO
[10/300] Suy luận cục bộ ASIN: B09CTPCB6M
[11/300] Suy luận cục bộ ASIN: B0026ZYZ7Q
[12/300] Suy luận cục bộ ASIN: B07D8KXWD1
[13/300] Suy luận cục bộ ASIN: B085TBVW4Y
[14/300] Suy luận cục bộ ASIN: B0991WM49R
[15/300] Suy luận cục bộ ASIN: B0060KPGDK
[16/300] Suy luận cục bộ ASIN: B09T3G8HT5
[17/300] Suy luận cục bộ ASIN: B002HZ5MRM
[18/300] Suy luận cục bộ ASIN: B0888SLGN4
[19/300] Suy luận cục bộ ASIN: B074MZC5MD
[20/300] Suy luận cục bộ ASIN: B094D1B6T1
[21/300] Suy luận cục bộ ASIN: B082KN1ZXS
[22/300] Suy luận cục bộ ASIN: B07WGF95PM
[23/300] Suy luận cục bộ ASIN: B078S4B2MP
[24/300] Suy luận cục bộ ASIN: B09C6218T7
[25/300] Suy luận cục bộ ASIN: B07Q2DP2S2
[26/300] Suy luận cục bộ ASIN: B07WL4ZH3T